# Week 2 Day 4 — Airline AI Assistant

This project extends the Day 4 Airline AI Assistant by replacing
hardcoded flight information with real-time flight data from the
Aviationstack API.

The assistant uses OpenAI function calling to decide when it needs
to search for flights.

For airport lookup, the project uses the `airportsdata` package,
allowing users to provide city names, airport names, or IATA codes.

## What are we building?

We are building an AI-powered airline assistant that can:

- Understand a user's flight-search request
- Convert city or airport names into IATA airport codes
- Search real-time flight information using Aviationstack
- Remove codeshare duplicates
- Convert flight times from UTC to IST
- Present the results in a natural, conversational response

## How to use the assistant

You can ask questions such as:

- "Find flights from Delhi to Mumbai"
- "Show me flights from Bhubaneswar to Delhi"
- "Are there flights from Mumbai to Bangalore?"
- "Find flights from DEL to BOM"

You can provide:

- City names
- Airport names
- IATA airport codes

The assistant returns information such as:

- Airline
- Flight number
- Departure airport
- Arrival airport
- Scheduled departure time
- Scheduled arrival time
- Flight status

> **Note:** This project provides flight information only.
> It does not provide ticket prices, seat availability, or booking functionality.

## How it works

```text
User
  ↓
OpenAI
  ↓
Function calling
  ↓
search_flights()
  ↓
airportsdata
  ↓
City/Airport → IATA code
  ↓
Aviationstack API
  ↓
Flight information
  ↓
Remove codeshare duplicates
  ↓
Convert UTC → IST
  ↓
OpenAI
  ↓
Natural-language response

## Requirements

Before running this notebook, make sure you have:

- Python 3.10 or later
- An OpenAI API key
- An Aviationstack API key
- An active internet connection

### Install the required packages

Run the following command if the packages are not already installed:

```bash
pip install openai python-dotenv requests airportsdata gradio


Then, separately, you already have the `.env` file in your project directory:

```text
OPENAI_API_KEY=your_openai_api_key
AVIATIONSTACK_API_KEY=your_aviationstack_api_key

In [38]:
import os
import json
import requests
import airportsdata

from datetime import datetime
from zoneinfo import ZoneInfo

from dotenv import load_dotenv
from openai import OpenAI
import gradio as gr

In [39]:
load_dotenv(override=True)

openai_api_key = os.getenv("OPENAI_API_KEY")
aviationstack_api_key = os.getenv("AVIATIONSTACK_API_KEY")

if openai_api_key:
    print("OpenAI API key loaded")
else:
    print("OpenAI API key not set")

if aviationstack_api_key:
    print("Aviationstack API key loaded")
else:
    print("Aviationstack API key not set")

MODEL = "gpt-4.1-mini"

openai = OpenAI()

OpenAI API key loaded
Aviationstack API key loaded


In [40]:
airports = airportsdata.load("IATA")

print(f"Loaded {len(airports):,} airports")

Loaded 7,884 airports


In [41]:
def get_iata_code(location):
    location = location.strip()

    # If the user already provided an IATA code
    if location.upper() in airports:
        return location.upper()

    location = location.lower()

    # Search by city, subdivision, or airport name
    for iata, airport in airports.items():
        if (
            airport["city"].lower() == location
            or airport["subd"].lower() == location
            or airport["name"].lower() == location
        ):
            return iata

    return None

In [42]:
test_locations = [
    "Delhi",
    "New Delhi",
    "Mumbai",
    "Bhubaneswar",
    "DEL"
]

for location in test_locations:
    print(f"{location:15} → {get_iata_code(location)}")

Delhi           → DEL
New Delhi       → DEL
Mumbai          → BOM
Bhubaneswar     → BBI
DEL             → DEL


In [43]:
def format_time(timestamp):
    dt = datetime.fromisoformat(timestamp)
    dt = dt.astimezone(ZoneInfo("Asia/Kolkata"))
    
    return dt.strftime("%d %b %Y, %I:%M %p IST")

In [44]:
def search_flights(source, destination):
    source_code = get_iata_code(source)
    destination_code = get_iata_code(destination)

    if not source_code:
        return f"Could not find an airport for '{source}'."

    if not destination_code:
        return f"Could not find an airport for '{destination}'."

    url = "http://api.aviationstack.com/v1/flights"

    params = {
        "access_key": aviationstack_api_key,
        "dep_iata": source_code,
        "arr_iata": destination_code,
        "limit": 10
    }

    try:
        response = requests.get(
            url,
            params=params,
            timeout=10
        )

        response.raise_for_status()
        data = response.json()

        if "error" in data:
            return (
                f"Aviationstack error: "
                f"{data['error'].get('message', 'Unknown error')}"
            )

        if not data.get("data"):
            return (
                f"No flights found from "
                f"{source_code} to {destination_code}."
            )

        flights = []

        for flight in data["data"]:

            # Ignore codeshare flights
            if flight.get("flight", {}).get("codeshared"):
                continue

            flights.append({
                "airline": flight["airline"]["name"],
                "flight_number": flight["flight"]["iata"],
                "source": flight["departure"]["iata"],
                "destination": flight["arrival"]["iata"],
                "departure": format_time(
                    flight["departure"]["scheduled"]
                ),
                "arrival": format_time(
                    flight["arrival"]["scheduled"]
                ),
                "status": flight["flight_status"]
            })

        if not flights:
            return (
                f"No operating flights found from "
                f"{source_code} to {destination_code}."
            )

        return json.dumps(flights, indent=2)

    except requests.RequestException as e:
        return f"Unable to retrieve flight information: {e}"

In [45]:
flight_search_function = {
    "name": "search_flights",
    "description": (
        "Search for scheduled flights between two locations "
        "using real-time Aviationstack data. "
        "The source and destination can be city names, "
        "airport names, or IATA airport codes."
    ),
    "parameters": {
        "type": "object",
        "properties": {
            "source": {
                "type": "string",
                "description": (
                    "Departure city, airport name, "
                    "or IATA airport code."
                )
            },
            "destination": {
                "type": "string",
                "description": (
                    "Arrival city, airport name, "
                    "or IATA airport code."
                )
            }
        },
        "required": ["source", "destination"],
        "additionalProperties": False
    }
}

tools = [
    {
        "type": "function",
        "function": flight_search_function
    }
]

In [46]:
def chat(message, history):
    messages = [
        {
            "role": "system",
            "content": (
                "You are FlightAI, a helpful airline assistant. "
                "Help users find flight information. "
                "Be concise, courteous, and accurate. "
                "You can search for flight information, but you cannot book tickets "
                "or provide ticket prices or seat availability. "
                "Do not claim to perform actions that are not supported."
        )
        }
    ]

    for user_message, assistant_message in history:
        messages.append({
            "role": "user",
            "content": user_message
        })

        messages.append({
            "role": "assistant",
            "content": assistant_message
        })

    messages.append({
        "role": "user",
        "content": message
    })

    response = openai.chat.completions.create(
        model=MODEL,
        messages=messages,
        tools=tools
    )

    assistant_message = response.choices[0].message

    if assistant_message.tool_calls:

        messages.append(assistant_message)

        for tool_call in assistant_message.tool_calls:

            if tool_call.function.name == "search_flights":

                arguments = json.loads(
                    tool_call.function.arguments
                )

                result = search_flights(
                    arguments["source"],
                    arguments["destination"]
                )

                messages.append({
                    "role": "tool",
                    "tool_call_id": tool_call.id,
                    "content": result
                })

        response = openai.chat.completions.create(
            model=MODEL,
            messages=messages
        )

    return response.choices[0].message.content

In [47]:
chat("Find flights from Delhi to Mumbai", [])

"I'm currently unable to retrieve live flight information due to a connectivity issue. However, you can usually find flights from Delhi (DEL) to Mumbai (BOM) operated by airlines such as Air India, IndiGo, and Vistara. For the latest flight schedules and availability, I recommend checking airline websites or popular travel booking platforms. Let me know if there's anything else I can assist you with!"

In [48]:
gr.ChatInterface(
    fn=chat,
    title="✈️ FlightAI",
    description=(
        "Ask me to find flights between cities or airports. "
        "You can use city names, airport names, or IATA codes."
    ),
    examples=[
        "Find flights from Delhi to Mumbai",
        "Show me flights from Bhubaneswar to Delhi",
        "Are there flights from Mumbai to Bangalore?",
        "Find flights from DEL to BOM"
    ]
).launch()

d:\genai\llm_engineering\.venv\Lib\site-packages\gradio\chat_interface.py:347: UserWarning: The 'tuples' format for chatbot messages is deprecated and will be removed in a future version of Gradio. Please set type='messages' instead, which uses openai-style 'role' and 'content' keys.
  self.chatbot = Chatbot(


* Running on local URL:  http://127.0.0.1:7861
* To create a public link, set `share=True` in `launch()`.
